<a href="https://colab.research.google.com/github/KDK-00/deeplearning/blob/main/news_api_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **News 추천 알고리즘**

사용 전 Newsapi.org 홈페이지에 접속하여

개인 api key를 발급받아야 합니다.



In [31]:
!pip install requests

해당하는 부분에 발급받은 api key를 입력해주세요

In [32]:
import requests

# API 키 설정
API_KEY = '630646c0b54e41abb999fe00886bfe2c'
BASE_URL = 'https://newsapi.org/v2/'

In [47]:
def get_top_headlines(api_key, country='kr', category='general', page_size=10):
    url = f"{BASE_URL}top-headlines"  # BASE_URL에 이미 '/v2/'가 포함되어 있음
    params = {
        'apiKey': api_key,
        'country': country,
        'category': category,
        'pageSize': page_size
    }
    response = requests.get(url, params=params)

    # Check for HTTP errors
    response.raise_for_status()  # Raise an exception for bad status codes

    return response.json()

# 한국의 비즈니스 뉴스 헤드라인 가져오기
headlines = get_top_headlines(API_KEY)

# 결과 출력
for article in headlines['articles']:
    print(f"Title: {article['title']}")
    print(f"Description: {article['description']}")
    print(f"URL: {article['url']}")
    print("-" * 50)

Title: 김규철 위원장 "밸브, 스팀 자체등급분류사업자 관심 보여" - 인벤
Description: 글로벌 게임 플랫폼 스팀(Steam)을 운영하는 밸브(Valve)가 국내 자체등급분류사업자가 되는 것에 관심을 두는 것으로 나타났다. 3일 게임물관리위원회 기자간담회 자리에서 스팀이 게임위 관리 영역 밖에 있어 확..
URL: https://www.inven.co.kr/webzine/news/?news=297165
--------------------------------------------------
Title: 상속세 '최대주주 할증' 폐지…배당 늘린 기업 법인세 감면 - 한겨레
Description: 
URL: 
--------------------------------------------------
Title: 대통령 “소상공인 지원, 현금 살포 아닌 구조적·항구적 대책 추진해야” - 대한민국정책포털 korea.kr
Description: 윤석열 대통령은 3일 “소상공인들이 위기를 극복하고 재기할 수 있도록 도움이 절실한 소상공인을 충분하게 지원하는 한편, 현금 살포와 같은 미봉책이 아니라 구조적이고 항구적인 대책을 함께 추진해야 한다”고 강조했다. 윤 대통령은 이날‘하반기 경제정책방향 및 역동경제 로드맵 발표’ 회의모두발언에서 “코로나19 시기에 대출을 받은 소상공인의 수와 대출 규모가 급
URL: https://www.korea.kr/news/policyNewsView.do?newsId=148931034
--------------------------------------------------
Title: "심장을 바쳐라" 진격의 거인 VR게임 출시일 확정 - 디스이즈게임
Description: VR 입체기동 좋았쓰!!
URL: https://m.thisisgame.com/webzine/news/nboard/4/?n=191084
--------------------------------------------------
Title: 이승윤, '마

In [49]:
def get_top_headlines(api_key, country='kr', category='business', page_size=100, max_pages=5):
    url = f"{BASE_URL}top-headlines"
    all_articles = []
    for page in range(1, max_pages + 1):
        params = {
            'apiKey': api_key,
            'country': country,
            'category': category,
            'pageSize': page_size,
            'page': page
        }
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        all_articles.extend(data['articles'])
        if len(data['articles']) < page_size:
            break  # 더 이상 기사가 없으면 중단
    return all_articles

# 한국의 비즈니스 뉴스 헤드라인 많이 가져오기
articles = get_top_headlines(API_KEY)

# 데이터 개수 확인
print(f"Total articles fetched: {len(articles)}")

Total articles fetched: 70


In [58]:
import re
import pandas as pd

def preprocess_articles(articles):
    cleaned_articles = []
    for article in articles:
        title = article['title'] if article['title'] else ''
        description = article['description'] if article['description'] else ''
        content = f"{title} {description}"
        content = re.sub(r'<[^>]+>', '', content)
        content = re.sub(r'[^a-zA-Z0-9가-힣\s]', '', content)
        cleaned_articles.append({
            'content': content.strip(),
            'source': article['source']['name'],
            'publishedAt': article['publishedAt']
        })
    return cleaned_articles

cleaned_articles = preprocess_articles(articles)

# DataFrame으로 정리
df = pd.DataFrame(cleaned_articles, columns=['content'])
print(df.head())

                                             content
0                 상속세 최대주주 할증 폐지배당 늘린 기업 법인세 감면  한겨레
1  대통령 소상공인 지원 현금 살포 아닌 구조적항구적 대책 추진해야  대한민국정책포털 ...
2  일본이 9900원 가격 실화냐버스 요금 보다 싼 항공권  한국경제 일본이 9900원...
3  초기 알츠하이머 신약  FDA 승인큰 기대보단 과도기로 생각해야  동아사이언스 새로...
4  올해 성장률 26 전망소상공인 지원에 25조 원 투입  KBS뉴스 앵커 정부가 올해...


In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def build_recommendation_model(cleaned_articles):
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(cleaned_articles)
    cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
    return cosine_sim

# 추천 모델 구축
cosine_sim = build_recommendation_model(cleaned_articles)

def recommend_articles(index, cosine_sim, df, top_n=5):
    sim_scores = list(enumerate(cosine_sim[index]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n + 1]

    article_indices = [i[0] for i in sim_scores]
    return df.iloc[article_indices]

# 첫 번째 기사와 유사한 기사 추천
recommended_articles = recommend_articles(0, cosine_sim, df)
print(recommended_articles)

                                              content
18  다시 불붙은 상속세 개편야당 반대 넘어설까  ZD넷 코리아 정부가 하반기 상속세 개...
48  855명만 월급 올려줘 300조 기업 흔드는 삼성전자 노조  서울경제신문 경제 18...
13  하나은행KB국민은행 주택담보대출 금리 인상  MBC 뉴스 가계대출 증가세를 늦추기 ...
45  LG엔솔 전기차 LFP 첫 수주중국 텃밭 뚫었다한국경제TV뉴스  한국경제TV뉴스 L...
37  르노 집게손 직원 징계 압박에명백한 인권 침해  한겨레 르노코리아의 새 차 홍보 영...


In [92]:
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import json

# **json형식으로 API 가져오기**

In [81]:
import requests
url = ('https://newsapi.org/v2/top-headlines?'
       'country=kr&'
       'category=business&'
       'apiKey=630646c0b54e41abb999fe00886bfe2c')
response = requests.get(url)
print(response.json())

{'status': 'ok', 'totalResults': 70, 'articles': [{'source': {'id': None, 'name': None}, 'author': None, 'title': "상속세 '최대주주 할증' 폐지…배당 늘린 기업 법인세 감면 - 한겨레", 'description': '', 'url': '', 'urlToImage': None, 'publishedAt': '2024-07-03T06:09:00Z', 'content': None}, {'source': {'id': None, 'name': 'Korea.kr'}, 'author': '대통령실', 'title': '대통령 “소상공인 지원, 현금 살포 아닌 구조적·항구적 대책 추진해야” - 대한민국정책포털 korea.kr', 'description': '윤석열 대통령은 3일 “소상공인들이 위기를 극복하고 재기할 수 있도록 도움이 절실한 소상공인을 충분하게 지원하는 한편, 현금 살포와 같은 미봉책이 아니라 구조적이고 항구적인 대책을 함께 추진해야 한다”고 강조했다. 윤 대통령은 이날‘하반기 경제정책방향 및 역동경제 로드맵 발표’ 회의모두발언에서 “코로나19 시기에 대출을 받은 소상공인의 수와 대출 규모가 급', 'url': 'https://www.korea.kr/news/policyNewsView.do?newsId=148931034', 'urlToImage': 'https://www.korea.kr/newsWeb/resources/attaches/2024.07/03/8833d37b167c7f553a9b9ea6c171b5d0.jpg', 'publishedAt': '2024-07-03T05:54:58Z', 'content': '.\r\n3 , .\r\n \xa0 \xa0\xa0 19 .\r\n, 19 , 19 .\r\n \xa0 \xa0 1 9 , 3 2% .\r\n, , .\r\n2020 2022 69 2022 , 42 .\r\n , , .\r\n .\r\n .\r\n , · .\r\n

In [93]:
json.loads(response.text)

{'status': 'ok',
 'totalResults': 70,
 'articles': [{'source': {'id': None, 'name': None},
   'author': None,
   'title': "상속세 '최대주주 할증' 폐지…배당 늘린 기업 법인세 감면 - 한겨레",
   'description': '',
   'url': '',
   'urlToImage': None,
   'publishedAt': '2024-07-03T06:09:00Z',
   'content': None},
  {'source': {'id': None, 'name': 'Korea.kr'},
   'author': '대통령실',
   'title': '대통령 “소상공인 지원, 현금 살포 아닌 구조적·항구적 대책 추진해야” - 대한민국정책포털 korea.kr',
   'description': '윤석열 대통령은 3일 “소상공인들이 위기를 극복하고 재기할 수 있도록 도움이 절실한 소상공인을 충분하게 지원하는 한편, 현금 살포와 같은 미봉책이 아니라 구조적이고 항구적인 대책을 함께 추진해야 한다”고 강조했다. 윤 대통령은 이날‘하반기 경제정책방향 및 역동경제 로드맵 발표’ 회의모두발언에서 “코로나19 시기에 대출을 받은 소상공인의 수와 대출 규모가 급',
   'url': 'https://www.korea.kr/news/policyNewsView.do?newsId=148931034',
   'urlToImage': 'https://www.korea.kr/newsWeb/resources/attaches/2024.07/03/8833d37b167c7f553a9b9ea6c171b5d0.jpg',
   'publishedAt': '2024-07-03T05:54:58Z',
   'content': '.\r\n3 , .\r\n \xa0 \xa0\xa0 19 .\r\n, 19 , 19 .\r\n \xa0 \xa0 1 9 , 3 2% .\r\n, , .\r\n2020 2022 69 2

In [94]:
json_data = pd.json_normalize(json.loads(response.text)['articles'])
json_data

,author,title,description,url,urlToImage,publishedAt,content,source.id,source.name
0,None,상속세 '최대주주 할증' 폐지…배당 늘린 기업 법인세 감면 - 한겨레,,,None,2024-07-03T06:09:00Z,None,None,None
1,대통령실,"대통령 “소상공인 지원, 현금 살포 아닌 구조적·항구적 대책 추진해야” - 대한민국...",윤석열 대통령은 3일 “소상공인들이 위기를 극복하고 재기할 수 있도록 도움이 절실한...,https://www.korea.kr/news/policyNewsView.do?ne...,https://www.korea.kr/newsWeb/resources/attache...,2024-07-03T05:54:58Z,".\r\n3 , .\r\n 19 .\r\n, 19 , 19 .\r\n ...",None,Korea.kr
2,김재후,"""일본이 9900원? 가격 실화냐""…버스 요금 보다 싼 항공권 - 한국경제","""일본이 9900원? 가격 실화냐""…버스 요금 보다 싼 항공권, 국내 1000원, ...",https://www.hankyung.com/article/202407039147i,https://img.hankyung.com/photo/202407/01.37231...,2024-07-03T04:56:51Z,"(LCC) . 1000 , 1 3 . 1000, ~ 9900 . 52900 ~ KT...",None,Hankyung.com
3,동아사이언스,"초기 알츠하이머 신약, 美 FDA 승인…""큰 기대보단 과도기로 생각해야"" - 동아사이언스",새로운 알츠하이머 치료제가 미국 내 사용 승인을 받았다. MARHARYTA MARK...,https://m.dongascience.com/news.php?idx=66286,https://image.dongascience.com/Photo/2024/07/1...,2024-07-03T03:55:00Z,. .\r\n (FDA) ‘(: )’ 2() . .\r\n ‘’ . . . \r...,None,Dongascience.com
4,김진화,“올해 성장률 2.6% 전망”…소상공인 지원에 25조 원 투입 - KBS뉴스,[앵커]<br /><br /> 정부가 올해 경제 성장률을 기존 2.2%에서 2.6%...,https://news.kbs.co.kr/news/view.do?ncd=8002614,http://news.kbs.co.kr/data/news/2024/07/03/202...,2024-07-03T03:43:00Z,"[]2.2% 2.6% .\r\n.\r\n.\r\n25 .\r\n, .\r\n[]\r...",None,Kbs.co.kr
5,최욱 기자,[하반기 경제] PF 자기자본 높인다…가계부채비율 90%대 초반 관리 - 연합인포맥스,정부가 우리 경제의 대표적인 잠재 리스크인 부동산 프로젝트파이낸싱(PF)과 가계부채...,https://news.einfomax.co.kr/news/articleView.h...,https://cdn.news.einfomax.co.kr/news/thumbnail...,2024-07-03T03:30:34Z,None,None,Einfomax.co.kr
6,조선일보,"현대차, 인도네시아에서 전기차 '코나' 생산...7억 인구 아세안 시장 잡는다 - ...","현대차, 인도네시아에서 전기차 코나 생산...7억 인구 아세안 시장 잡는다",https://www.chosun.com/economy/auto/2024/07/03...,https://images.chosun.com/resizer/_ZMX0yBdB0Dk...,2024-07-03T03:27:42Z,None,None,Chosun.com
7,조선일보,7월의 과학기술인상에 노준석 포스텍 교수…메타렌즈 대량 생산 성공 - 조선일보,7월의 과학기술인상에 노준석 포스텍 교수메타렌즈 대량 생산 성공,https://www.chosun.com/economy/science/2024/07...,https://images.chosun.com/resizer/yabhmnczBpZi...,2024-07-03T03:04:26Z,None,None,Chosun.com
8,김인엽,부동산에 주식까지 다 내다 판다…돈 싸들고 떠나는 부자들 - 한국경제,"부동산에 주식까지 다 내다 판다…돈 싸들고 떠나는 부자들, 총선 리스크에 떠는 영국...",https://www.hankyung.com/article/202407038289i,https://img.hankyung.com/photo/202407/99.37231...,2024-07-03T02:31:12Z,"= \r\n() . 4() . . . /\r\n3 (FT) · (CEO) . "" ""...",None,Hankyung.com
9,안선영,아플 때는 늦다…중산층 10명 중 8명은 상속 준비 필요 - 아주경제,# 50대 남성 K씨는 최근 상속에 대한 고민을 시작했다. 서울 성동구에 10억원대...,https://www.ajunews.com/view/20240703105645555,https://image.ajunews.com/content/image/2024/0...,2024-07-03T02:07:35Z,"# 50 K . 10 , 34 . 10 . . . K .\r\n3 ' ' .\r\n...",None,Ajunews.com


1